# IMPORTAÇÕES

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# DATASET - 01_canais_venda_sujo.csv

In [2]:
df = pd.read_csv('01_canais_venda_sujo.csv', sep=';')

print(df.shape)

df.head(20)

(4, 3)


,id_canal,nome_canal,comissao_pct
0,1,Site Próprio,8%
1,2,ota,18%
2,3,Telefone,5%
3,4,agencia,12%


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   id_canal      4 non-null      int64
 1   nome_canal    4 non-null      str  
 2   comissao_pct  4 non-null      str  
dtypes: int64(1), str(2)
memory usage: 228.0 bytes


In [4]:
# Tipos de cada coluna
display(df.dtypes)

# Estatísticas das colunas numéricas
display(df.describe())

# Colunas de texto também
display(df.describe(include='object'))

id_canal        int64
nome_canal        str
comissao_pct      str
dtype: object

,id_canal
count,4.000000
mean,2.500000
std,1.290994
min,1.000000
25%,1.750000
50%,2.500000
75%,3.250000
max,4.000000


C:\Users\rodri\AppData\Local\Temp\ipykernel_13600\106439391.py:8: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  display(df.describe(include='object'))


,nome_canal,comissao_pct
count,4,4
unique,4,4
top,Site Próprio,8%
freq,1,1


# AUDITORIA - 01_canais_venda_sujo.csv

In [5]:
# Contagem de nulos por coluna
df.isnull().sum()

id_canal        0
nome_canal      0
comissao_pct    0
dtype: int64

In [6]:
# Percentual de nulos — colunas com >15% exigem atenção especial
(df.isnull().sum() / len(df) * 100).round(2)

id_canal        0.0
nome_canal      0.0
comissao_pct    0.0
dtype: float64

In [7]:
# Ver QUAIS linhas têm pelo menos um nulo
df[df.isnull()     # Gera True/False para cada dado (True = nulo)
   .any(axis=1)]   # any(): existe pelo menos um True? axis=1: por linha

,id_canal,nome_canal,comissao_pct


In [8]:
# Quantidade de duplicatas
df.duplicated().sum()

np.int64(0)

In [9]:
# Visualizar TODAS as ocorrências duplicadas
df[df.duplicated(keep=False)]

,id_canal,nome_canal,comissao_pct


In [10]:
# Duplicata apenas por colunas-chave

df.duplicated(subset=['nome_canal','comissao_pct']).sum()


##### verificar utilidade csv de 5 linhas somente

np.int64(0)

# Inconsistências de Texto: value_counts()

In [11]:
# Ver variações do TEXTO
df['nome_canal'].value_counts()
# Saída esperada:
# TI           12
# ti            5  ← mesmo valor!
# T.I.          3  ← mesmo valor!
# Tecnologia    4  ← mesmo valor!

nome_canal
Site Próprio    1
ota             1
Telefone        1
agencia         1
Name: count, dtype: int64

In [12]:
# Outras colunas para checar:
print('--- nome_canal ---')
print(df['nome_canal'].value_counts())

print('\n--- comissao_pct ---')
print(df['comissao_pct'].value_counts())

# nome: verificar espaços invisíveis
print('\n--- canais com espaço nas bordas ---')
mascara = df['nome_canal'].str.startswith(' ') | df['nome_canal'].str.endswith(' ')
print(df.loc[mascara, 'nome_canal'].tolist())




#### OTA , ONLINE TRAVEL AGENCY DEVE SER EM MAIUSCULO;


--- nome_canal ---
nome_canal
Site Próprio    1
ota             1
Telefone        1
agencia         1
Name: count, dtype: int64

--- comissao_pct ---
comissao_pct
8%     1
18%    1
5%     1
12%    1
Name: count, dtype: int64

--- canais com espaço nas bordas ---
[]


# Relatório de Auditoria — Resumo dos Problemas

In [13]:
# Guardar shape inicial para o relatório final
shape_inicial = df.shape[0]
print(f'Registros iniciais: {shape_inicial}')

Registros iniciais: 4


In [14]:
# Remover duplicatas completas
df = df.drop_duplicates()
print(df.shape)

(4, 3)


In [15]:
# Remover
print(df.shape)  # antes
df = df.drop_duplicates()
print(df.shape)  # depois


(4, 3)
(4, 3)


In [16]:
# Duplicata por colunas específicas (quando necessário)
# df = df.drop_duplicates(
#     subset=['nome','departamento'],
#     keep='first')

## Tratando Nulos: dropna() vs fillna()


In [17]:
df = df.dropna(subset=['id_canal'])
# df = df.dropna(thresh=6)  # remove linhas com menos de 6 valores não-nulos

df['id_canal'] = df['id_canal'].fillna('sem_id_canal')
# df['salario'] = df['salario'].fillna(mediana)  # será feito após limpeza do tipo
print(df) # SO PARA VER SE O FILLNA FUNCIONOU MESMO, NÃO VALE EM TEXTOS GRANDES.

   id_canal    nome_canal comissao_pct
0         1  Site Próprio           8%
1         2           ota          18%
2         3      Telefone           5%
3         4       agencia          12%


## Corrigindo Tipos: strings, números e datas

Dados com tipo errado impedem cálculos, filtros e visualizações corretos.  
**Sempre limpar ANTES de converter.**

`salario` — limpeza antes da conversão:
```
"3.500,00"  →  str.replace(',','.')  →  "3.500.00"  →  pd.to_numeric()  →  3500.0
     str                                    str                              float
```


In [18]:
# Passo 1: limpar conteúdo 
df['comissao_pct'] = df['comissao_pct'].astype(str).str.replace('%', '').str.replace(',', '.')

# Passo 2: converter tipo
df['comissao_pct'] = pd.to_numeric(df['comissao_pct'], errors='coerce')

print('Tipo:', df['comissao_pct'].dtype)
print('Nulos gerados:', df['comissao_pct'].isnull().sum())

print('\n')
print(df) # SO PARA VER SE  FUNCIONOU, NÃO VALE EM ARQUIVOS GRANDES.

Tipo: int64
Nulos gerados: 0


   id_canal    nome_canal  comissao_pct
0         1  Site Próprio             8
1         2           ota            18
2         3      Telefone             5
3         4       agencia            12


In [19]:
mediana_comissao_pct = df['comissao_pct'].median()
#print('\n')
print(f'Mediana da comissão calculada com dados limpos: {mediana_comissao_pct:,.2f}%')
df['comissao_pct'] = df['comissao_pct'].fillna(mediana_comissao_pct)



Mediana da comissão calculada com dados limpos: 10.00%


## Padronizando Texto: str.strip(), str.upper(), map()

| Método | O que faz |
|---|---|
| `.str.strip()` | Remove espaços nas bordas |
| `.str.lower()` | Converte para minúsculas |
| `.str.upper()` | Converte para maiúsculas |
| `.str.title()` | Primeira letra de cada palavra maiúscula |
| `.str.replace()` | Substitui substring por outra |
| `.str.contains()` | Filtra linhas que contêm padrão |

`fillna(df['departamento'])` — mantém os valores que já estavam corretos e não precisavam de remapeamento.


In [20]:
# Espaços e capitalização — sempre primeiro
df['nome_canal'] = df['nome_canal'].str.strip()
df['nome_canal'] = df['nome_canal'].str.strip().str.upper()

# Ver o que sobrou após padronizar
print('Valores únicos em nome_canal:')
print(sorted(df['nome_canal'].unique()))

Valores únicos em nome_canal:
['AGENCIA', 'OTA', 'SITE PRÓPRIO', 'TELEFONE']


In [21]:
# Mapear variações para o valor correto
mapa = {
    'AGENCIA':                     'AGÊNCIA',
   }

df['nome_canal'] = df['nome_canal'].map(mapa).fillna(df['nome_canal'])

print('Após mapeamento:')
print(sorted(df['nome_canal'].unique()))

Após mapeamento:
['AGÊNCIA', 'OTA', 'SITE PRÓPRIO', 'TELEFONE']


---
# 🔧 Bloco 4 · Tratamento



## Valores Impossíveis: filtro com .loc e between()

Dados que passaram pela validação do sistema mas não fazem sentido no mundo real.

**Sempre registre quantos registros foram removidos em cada etapa** — é parte da auditoria e deve ser documentado.  
`between(a,b)` é equivalente a `(df['col'] >= a) & (df['col'] <= b)` — mais legível.


## NÃO POSSUI VARIAÇÕES PARA MAPEAR, APENAS AGÊNCIA COM A VERSÃO SEM ACENTO.

## Detectando e Tratando Outliers: IQR

**O que fazer com outliers? Três opções:**

| Decisão | Quando usar |
|---|---|
| **Remover** | Valor claramente inválido (ex: salário 999999 — erro de digitação) |
| **Substituir** | Registro válido, mas valor improvável — substitua pela mediana |
| **Manter** | Valor real e relevante (ex: salário de diretor — cliente VIP) |


## NÃO POSSUI OUTLIERS, NÃO FEITO

## Colunas Derivadas: apply() e Operações Vetorizadas

Criar novas colunas a partir dos dados existentes enriquece o dataset para análise.  
**Operações vetorizadas (sem apply) são 10–100× mais rápidas** — use sempre que possível.  
Use `apply()` só quando precisar de lógica condicional.



## NÃO FEITO, NÃO VI CRITERIO PARA ELE 


---
# ✅ Bloco 5 · Resultado Final


## Dataset Antes vs. Depois — Comparativo Final

| Problema Identificado | Solução Aplicada | Etapa |
|---|---|---|
| 10 linhas duplicadas | `drop_duplicates()` | Limpeza |
| 8 nulos em email | `dropna(subset=['email'])` | Limpeza |
| `salario` como string com aspas e vírgula | `str.replace()` + `pd.to_numeric()` | Limpeza |
| `data_admissao` com 3 formatos mistos | `pd.to_datetime(format='mixed')` | Limpeza |
| `departamento` com 5+ variações | `str.strip()` + `str.upper()` + `map()` | Limpeza |
| `nome` com espaços invisíveis | `str.strip()` | Limpeza |
| 3 salários negativos | filtro `.loc > 0` | Tratamento |
| 1 outlier extremo (999999) | IQR — remoção | Tratamento |
| Sem coluna `faixa_salarial` | `apply(faixa_salarial)` | Tratamento |


In [22]:
print('=' * 48)
print('      RELATÓRIO FINAL DE AUDITORIA')
print('=' * 48)
print(f'Registros iniciais:     {shape_inicial:>6}')
print(f'Registros finais:       {df.shape[0]:>6}')
print(f'Registros removidos:    {shape_inicial - df.shape[0]:>6}')
print(f'Colunas originais:      {8:>6}')
print(f'Colunas finais:         {df.shape[1]:>6}')
print('=' * 48)
print(f'Nulos restantes:        {df.isnull().sum().sum():>6}')
print(f'Duplicatas restantes:   {df.duplicated().sum():>6}')
print('=' * 48)

      RELATÓRIO FINAL DE AUDITORIA
Registros iniciais:          4
Registros finais:            4
Registros removidos:         0
Colunas originais:           8
Colunas finais:              3
Nulos restantes:             0
Duplicatas restantes:        0


In [23]:
display(df.head(10))

,id_canal,nome_canal,comissao_pct
0,1,SITE PRÓPRIO,8
1,2,OTA,18
2,3,TELEFONE,5
3,4,AGÊNCIA,12


## EXPORTAÇÃO DA BASE SUJA PARA LIMPA

In [24]:
# df.to_csv('03_canais_venda_clean.csv',
#  sep=';', index=False, encoding='utf-8') # ORIGINAL DA ERRO NO EXCEL, POR CAUSA DO BOMBEAMENTO DE ACENTOS

# UTF-8 COM BOMBEAMENTO (BOM) PARA GARANTIR COMPATIBILIDADE COM EXCEL
df.to_csv('03_canais_venda_clean.csv', sep=';', index=False, encoding='utf-8-sig')
print('✅ Base exportada: 03_canais_venda_clean.csv')

print('\n')
print(df.info())
print('\n')
print(df)

✅ Base exportada: 03_canais_venda_clean.csv


<class 'pandas.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   id_canal      4 non-null      int64
 1   nome_canal    4 non-null      str  
 2   comissao_pct  4 non-null      int64
dtypes: int64(2), str(1)
memory usage: 228.0 bytes
None


   id_canal    nome_canal  comissao_pct
0         1  SITE PRÓPRIO             8
1         2           OTA            18
2         3      TELEFONE             5
3         4       AGÊNCIA            12


## Checklist de Auditoria e Limpeza

Use este checklist em qualquer projeto de análise de dados com Pandas.

### Auditoria
- [ ] `df.shape` → quantas linhas e colunas?
- [ ] `df.info()` → tipos corretos? há nulos?
- [ ] `df.describe()` → outliers numéricos?
- [ ] `df.isnull().sum()` → quais colunas têm nulos?
- [ ] `df.duplicated().sum()` → linhas repetidas?
- [ ] `df['col'].value_counts()` → inconsistências de texto?

### Limpeza & Tratamento
- [ ] Remover duplicatas (verificar contexto antes!)
- [ ] Limpar strings **ANTES** de converter tipos
- [ ] Converter tipos: `astype`, `to_numeric`, `to_datetime`
- [ ] Padronizar texto: `strip`, `upper/lower`, `map()`
- [ ] Tratar nulos com critério: `dropna` ou `fillna`
- [ ] Tratar outliers: remover, substituir ou manter
- [ ] Criar colunas derivadas para enriquecer análise
